<div style="background:#161920;color:#F7FAFC;padding:32px 36px;border-radius:16px;border-top:5px solid #35B1B7;font-family:Arial,sans-serif">
  <div style="color:#35B1B7;font-size:15px;font-weight:700;letter-spacing:.08em">PYTORCH PRACTICAL · SEQUENCE MODELLING</div>
  <h1 style="font-size:38px;margin:10px 0 8px">From RNN to LSTM: Forecast the Next Value</h1>
  <p style="font-size:18px;line-height:1.55;margin:0;color:#D9E2EC">Turn a time series into supervised sequences, train two recurrent networks under the same conditions, and explain when memory gates help.</p>
</div>

## Learning outcomes

By the end of this practical, you will be able to:

1. explain why ordinary feed-forward models do not naturally preserve sequence order;
2. convert a time series into input windows and next-step targets;
3. trace the tensor shapes entering and leaving `nn.RNN` and `nn.LSTM`;
4. train both models with the same leakage-safe chronological split;
5. compare MAE, RMSE and MAPE on unseen months; and
6. generate an autoregressive multi-step forecast and discuss error accumulation.

**Practical flow:** data → chronological split → scaling → windows → RNN → LSTM → comparison → future forecast.

### Before you run

In **Google Colab**, choose **Runtime → Run all**. A CPU runtime is sufficient; no file upload or Google Drive mount is required. The notebook downloads the monthly airline-passenger series automatically. If the classroom network blocks that URL, the same 144 observations are available as a built-in fallback.

Approximate runtime: **2–4 minutes on CPU**. Random seeds are fixed for repeatability, although small numerical differences across hardware are normal.

## 1 · Set up a reproducible PyTorch session

**What we do.** Check the few packages used in the practical, import them, select the available device and fix random seeds.

**Why.** A teaching notebook should run from a fresh Colab session and produce results that are similar enough for a class discussion.

In [ ]:
import importlib.util, subprocess, sys

required = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "torch": "torch"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from pathlib import Path
from copy import deepcopy
import io, math, random, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, max(1, torch.get_num_threads())))
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {"actual": "#172A3A", "rnn": "#E07A5F", "lstm": "#35B1B7", "future": "#F2B134"}
print(f"PyTorch {torch.__version__} | device: {device}")

## 2 · Frame the business problem as sequence prediction

We observe monthly international-airline passenger totals from 1949–1960. At month $t$, the model receives the previous **12 months** and predicts month $t+1$.

A recurrent model updates a hidden state as it reads the sequence:

$$h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b), \qquad \hat{y}_{t+1}=W_{hy}h_t+b_y.$$

The hidden state is a learned summary of what the network has seen. A plain RNN can model short dependencies, but repeated multiplication through many steps can make gradients shrink or grow. The LSTM introduces gates and a cell state to regulate what is remembered, written and exposed.

### Download and inspect the series

**What we do.** Fetch a two-column CSV (`Month`, `Passengers`) and parse the month as a date.

**Why.** Dates let us preserve temporal order. We never randomly shuffle observations before splitting: doing so would allow future patterns to leak into training.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
FALLBACK_VALUES = [
    112,118,132,129,121,135,148,148,136,119,104,118,
    115,126,141,135,125,149,170,170,158,133,114,140,
    145,150,178,163,172,178,199,199,184,162,146,166,
    171,180,193,181,183,218,230,242,209,191,172,194,
    196,196,236,235,229,243,264,272,237,211,180,201,
    204,188,235,227,234,264,302,293,259,229,203,229,
    242,233,267,269,270,315,364,347,312,274,237,278,
    284,277,317,313,318,374,413,405,355,306,271,306,
    315,301,356,348,355,422,465,467,404,347,305,336,
    340,318,362,348,363,435,491,505,404,359,310,337,
    360,342,406,396,420,472,548,559,463,407,362,405,
    417,391,419,461,472,535,622,606,508,461,390,432
]

try:
    with urllib.request.urlopen(DATA_URL, timeout=20) as response:
        df = pd.read_csv(io.BytesIO(response.read()))
    source = "automatic web download"
except Exception as exc:
    df = pd.DataFrame({
        "Month": pd.date_range("1949-01-01", periods=len(FALLBACK_VALUES), freq="MS").strftime("%Y-%m"),
        "Passengers": FALLBACK_VALUES,
    })
    source = f"embedded fallback ({type(exc).__name__})"

df["Month"] = pd.to_datetime(df["Month"])
df["Passengers"] = df["Passengers"].astype("float32")
assert len(df) == 144 and df["Passengers"].notna().all()
print(f"Data source: {source} | observations: {len(df)}")
display(df.head())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(df["Month"], df["Passengers"], color=COLORS["actual"], linewidth=2.2)
ax.set(title="Monthly international airline passengers", xlabel="Month", ylabel="Passengers (thousands)")
ax.margins(x=0.01)
plt.show()

**How to read the figure.** The upward trend means the level changes over time; the repeating peaks reveal yearly seasonality. A 12-month input window therefore gives each model one complete seasonal cycle. The growing peak-to-trough range also warns us that a simple model may underpredict later, larger values.

## 3 · Split first, then scale, then build windows

**What we do.** Reserve the first 70% for training, the next 15% for validation and the final 15% for testing. Mean and standard deviation are estimated **only from the training period**.

**Why.** Scaling with all 144 observations would reveal the future test distribution. Validation selects a training checkpoint; the test period remains untouched until final evaluation.

For a window length $L=12$:

$$X_i=[y_i,y_{i+1},\ldots,y_{i+11}], \qquad Y_i=y_{i+12}.$$

In [ ]:
values = df["Passengers"].to_numpy(dtype=np.float32)
dates = df["Month"].to_numpy()
n = len(values)
train_end = int(0.70 * n)
val_end = int(0.85 * n)
WINDOW = 12

train_mean = values[:train_end].mean()
train_std = values[:train_end].std()
scaled = (values - train_mean) / train_std

def make_windows(series, window):
    X, y, target_indices = [], [], []
    for target_index in range(window, len(series)):
        X.append(series[target_index-window:target_index])
        y.append(series[target_index])
        target_indices.append(target_index)
    X = torch.tensor(np.asarray(X), dtype=torch.float32).unsqueeze(-1)
    y = torch.tensor(np.asarray(y), dtype=torch.float32).unsqueeze(-1)
    return X, y, np.asarray(target_indices)

X_all, y_all, target_idx = make_windows(scaled, WINDOW)
train_mask = target_idx < train_end
val_mask = (target_idx >= train_end) & (target_idx < val_end)
test_mask = target_idx >= val_end

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

print(f"Train/validation/test targets: {len(y_train)}/{len(y_val)}/{len(y_test)}")
print(f"One batch input shape: {next(iter(train_loader))[0].shape} = [batch, time, features]")

In [ ]:
example = 0
input_values = X_train[example, :, 0].numpy() * train_std + train_mean
target_value = y_train[example, 0].item() * train_std + train_mean
example_dates = dates[:WINDOW]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(example_dates, input_values, marker="o", color=COLORS["actual"], label="12 input months")
ax.scatter(dates[WINDOW], target_value, s=100, color=COLORS["future"], zorder=3, label="next-month target")
ax.axvline(dates[WINDOW], color="#8896A6", linestyle="--", linewidth=1)
ax.set(title="One supervised learning example", xlabel="Month", ylabel="Passengers (thousands)")
ax.legend()
plt.show()

**How to read the figure.** The connected points are one input sequence. The isolated point is the label. Training does not ask the network to reproduce every input; it asks for the single value immediately after the window.

## 4 · Model A: a vanilla RNN

**What we do.** Feed one scalar per time step to `nn.RNN`, retain the output from the final time step, and map it to one forecast with a linear layer.

**Why.** This is the simplest learned recurrent baseline. `batch_first=True` keeps tensors in the readable form `[batch, time, feature]`.

| Tensor | Shape | Meaning |
|---|---:|---|
| Input | `[B, 12, 1]` | 12 scaled passenger values |
| RNN output | `[B, 12, H]` | hidden representation at every step |
| Final state | `[B, H]` | summary after month 12 |
| Forecast | `[B, 1]` | next scaled value |

In [ ]:
class RNNForecaster(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.recurrent = nn.RNN(
            input_size=1, hidden_size=hidden_size,
            num_layers=1, batch_first=True, nonlinearity="tanh"
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        sequence_output, final_hidden = self.recurrent(x)
        last_output = sequence_output[:, -1, :]
        return self.head(last_output)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

rnn_model = RNNForecaster().to(device)
sample_output = rnn_model(X_train[:4].to(device))
print(rnn_model)
print(f"Trainable parameters: {count_parameters(rnn_model):,}")
print(f"Shape check: {tuple(X_train[:4].shape)} → {tuple(sample_output.shape)}")

### Use one fair training procedure for both models

**What we do.** Minimize mean-squared error with Adam, clip gradients, and retain the checkpoint with the lowest validation loss.

**Why.** Gradient clipping limits unstable recurrent updates. Early stopping prevents us from choosing the final epoch merely because it ran last. Reusing this function makes the RNN/LSTM comparison meaningful.

In [ ]:
@torch.no_grad()
def mean_loss(model, loader, criterion):
    model.eval()
    total, count = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        total += criterion(model(xb), yb).item() * len(xb)
        count += len(xb)
    return total / count

def fit(model, train_loader, val_loader, max_epochs=250, learning_rate=1e-2, patience=30):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train": [], "val": []}
    best_state, best_val, stale = None, float("inf"), 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        running, count = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running += loss.item() * len(xb)
            count += len(xb)

        train_loss = running / count
        val_loss = mean_loss(model, val_loader, criterion)
        history["train"].append(train_loss)
        history["val"].append(val_loss)

        if val_loss < best_val - 1e-5:
            best_val, stale = val_loss, 0
            best_state = deepcopy(model.state_dict())
        else:
            stale += 1

        if epoch == 1 or epoch % 25 == 0:
            print(f"epoch {epoch:3d} | train {train_loss:.4f} | validation {val_loss:.4f}")
        if stale >= patience:
            print(f"early stopping at epoch {epoch}; best validation loss {best_val:.4f}")
            break

    model.load_state_dict(best_state)
    return history

rnn_history = fit(rnn_model, train_loader, val_loader)

## 5 · Model B: an LSTM

An LSTM carries a hidden state $h_t$ and a cell state $c_t$. Three sigmoid gates return values between 0 and 1:

- **forget gate**: how much old cell memory to retain;
- **input gate**: how much candidate information to write; and
- **output gate**: how much cell information to expose as the hidden state.

A simplified update is:

$$c_t=f_t\odot c_{t-1}+i_t\odot \tilde{c}_t, \qquad h_t=o_t\odot\tanh(c_t).$$

**Why this may help.** The additive cell-state path gives gradients a less fragile route across longer spans. It does not guarantee that the LSTM wins on every small dataset; that is an empirical question we now test.

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.recurrent = nn.LSTM(
            input_size=1, hidden_size=hidden_size,
            num_layers=1, batch_first=True
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        sequence_output, (final_hidden, final_cell) = self.recurrent(x)
        last_output = sequence_output[:, -1, :]
        return self.head(last_output)

# Reset the seed so model initialization is reproducible.
torch.manual_seed(SEED)
lstm_model = LSTMForecaster().to(device)
print(lstm_model)
print(f"Trainable parameters: {count_parameters(lstm_model):,}")
lstm_history = fit(lstm_model, train_loader, val_loader)

## 6 · Compare both models on the untouched test period

**What we do.** Convert predictions back to passenger counts and report three complementary errors.

- **MAE**: average absolute error in thousands of passengers; easy to explain.
- **RMSE**: penalizes large misses more strongly than MAE.
- **MAPE**: average percentage error; scale-free, but unsafe when actual values are zero (not a problem here).

Lower is better for all three. A persistence baseline—“next month equals this month”—keeps the neural results honest.

In [ ]:
@torch.no_grad()
def predict_scaled(model, X):
    model.eval()
    return model(X.to(device)).cpu().numpy().ravel()

def invert(z):
    return z * train_std + train_mean

def regression_metrics(actual, predicted):
    error = predicted - actual
    return {
        "MAE": np.mean(np.abs(error)),
        "RMSE": np.sqrt(np.mean(error ** 2)),
        "MAPE (%)": np.mean(np.abs(error / actual)) * 100,
    }

actual = invert(y_test.numpy().ravel())
rnn_pred = invert(predict_scaled(rnn_model, X_test))
lstm_pred = invert(predict_scaled(lstm_model, X_test))
persistence_pred = invert(X_test[:, -1, 0].numpy())

results = pd.DataFrame({
    "Persistence": regression_metrics(actual, persistence_pred),
    "RNN": regression_metrics(actual, rnn_pred),
    "LSTM": regression_metrics(actual, lstm_pred),
}).T.round(2)
display(results.style.highlight_min(axis=0, color="#BDEDE0"))

In [ ]:
test_dates = dates[target_idx[test_mask]]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(rnn_history["train"], color=COLORS["rnn"], label="RNN train")
axes[0].plot(rnn_history["val"], color=COLORS["rnn"], linestyle="--", label="RNN validation")
axes[0].plot(lstm_history["train"], color=COLORS["lstm"], label="LSTM train")
axes[0].plot(lstm_history["val"], color=COLORS["lstm"], linestyle="--", label="LSTM validation")
axes[0].set(title="Learning curves", xlabel="Epoch", ylabel="Mean-squared error (scaled)")
axes[0].set_yscale("log")
axes[0].legend()

axes[1].plot(test_dates, actual, marker="o", color=COLORS["actual"], linewidth=2.5, label="Actual")
axes[1].plot(test_dates, rnn_pred, marker="o", color=COLORS["rnn"], label="RNN")
axes[1].plot(test_dates, lstm_pred, marker="o", color=COLORS["lstm"], label="LSTM")
axes[1].set(title="Forecasts on unseen months", xlabel="Month", ylabel="Passengers (thousands)")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend()

plt.tight_layout()
plt.show()

**How to read the outputs.** The table answers “how wrong?”; the right-hand plot answers “when and in what direction?” A model can have a respectable average error while systematically missing seasonal peaks. In the learning curves, a falling training loss with a rising validation loss is a classic overfitting signal. The best model is the one with the strongest test evidence—not automatically the most complex architecture.

## 7 · Forecast 12 months beyond the dataset

**What we do.** Predict one month, append that prediction to the window, and repeat 12 times.

**Why.** This is autoregressive forecasting. It supports an arbitrary horizon without retraining, but later forecasts depend on earlier predicted—not observed—values. Errors can therefore compound.

In [ ]:
@torch.no_grad()
def forecast_future(model, history_scaled, window=12, horizon=12):
    model.eval()
    rolling = list(history_scaled[-window:])
    predictions = []
    for _ in range(horizon):
        x = torch.tensor(rolling[-window:], dtype=torch.float32).view(1, window, 1).to(device)
        next_value = model(x).item()
        predictions.append(next_value)
        rolling.append(next_value)
    return invert(np.asarray(predictions))

future = forecast_future(lstm_model, scaled, horizon=12)
future_dates = pd.date_range(df["Month"].iloc[-1] + pd.offsets.MonthBegin(1), periods=12, freq="MS")

fig, ax = plt.subplots(figsize=(13, 4.8))
ax.plot(df["Month"].iloc[-36:], values[-36:], color=COLORS["actual"], linewidth=2.3, label="Observed")
ax.plot(future_dates, future, marker="o", color=COLORS["future"], linewidth=2.3, label="LSTM autoregressive forecast")
ax.axvline(df["Month"].iloc[-1], color="#8896A6", linestyle="--")
ax.set(title="Twelve-step forecast: predictions become future inputs", xlabel="Month", ylabel="Passengers (thousands)")
ax.legend()
plt.show()

**How to read the figure.** Everything right of the dashed line is model-generated. Treat the trajectory as a teaching demonstration, not a production forecast: there are no uncertainty intervals, external drivers or model-updating steps. Notice whether the seasonal shape and rising level remain plausible as predictions feed back into the model.

## 8 · Guided exercises for the practical session

Work in pairs. Change **one factor at a time**, rerun from Section 3, and record the three test metrics.

1. **Memory span:** compare `WINDOW = 6`, `12` and `24`. Which choice contains a complete seasonal cycle?
2. **Capacity:** compare `hidden_size=8`, `32` and `64`. When does extra capacity stop helping validation loss?
3. **Depth:** add `num_layers=2`. If you add dropout, remember that recurrent dropout is applied *between stacked layers*, so it has no effect with one layer.
4. **Stress test:** train using only the first 50% of observations. Which architecture becomes less stable with less data?
5. **Communication:** explain to a manager why the lowest training loss is not the selection criterion.

**Expected reasoning—not a fixed winner.** A 12-month window matches the visible seasonal cycle. Larger networks can fit training data more closely but may generalize worse. The exact RNN/LSTM ranking may change with the split and seed; your explanation should use validation/test evidence.

## 9 · Knowledge check

1. Why must the split be chronological?  
   **Answer:** random splitting leaks future regimes and seasonal context into training.
2. Why fit the scaler on training data only?  
   **Answer:** validation/test statistics are information from the future.
3. What does the LSTM cell state add?  
   **Answer:** a gated, additive memory path that can preserve information and gradients across more steps.
4. Why can a 12-step recursive forecast degrade?  
   **Answer:** every new input contains earlier prediction error, so uncertainty accumulates.

### Source notes

- PyTorch APIs: [`nn.RNN`](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html) and [`nn.LSTM`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html).
- The classic monthly airline-passenger series is commonly attributed to Box & Jenkins and is downloaded here from the public `jbrownlee/Datasets` mirror.

## What you built

A complete, leakage-aware sequence-forecasting workflow: automatic data acquisition, chronological validation, train-only scaling, sliding windows, two recurrent architectures, early stopping, interpretable metrics, learning curves and an autoregressive future forecast. More importantly, you now have an evidence-based way to explain **why** an LSTM may help and **how** to test that claim fairly.